In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

### Step1- We load the Data and taking a look over it.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
df= pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
df

In [ ]:
df.columns

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
X= df.drop('SalePrice', axis=1)
y= df.SalePrice

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns

# ========== 2. Pipelines للأعمدة ==========
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

# ========== 3. النموذج ==========
model = Pipeline([
    ('preprocessor', preprocessor),
    ('rf', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
])

model.fit(X_train, y_train)

print("Train R²:", model.score(X_train, y_train))
print("Test  R²:", model.score(X_test, y_test))

# ========= 4. Permutation Importance ==========

# نجهّز نسخة X_test بعد التحويل (بدون fit!)
X_test_transformed = model.named_steps['preprocessor'].transform(X_test)

# إذا الناتج sparse matrix، نحوله إلى dense
if hasattr(X_test_transformed, "toarray"):
    X_test_transformed = X_test_transformed.toarray()

# نجيب أسماء الأعمدة بعد الـ ColumnTransformer
feature_names = model.named_steps['preprocessor'].get_feature_names_out()

# نحول الأسماء لتكون أنظف: نشيل البادئات "num__" و "cat__"
clean_names = [name.split("__")[-1] for name in feature_names]

# نحسب الـ permutation importance على الـ rf مباشرة
result = permutation_importance(
    model.named_steps['rf'], X_test_transformed, y_test,
    n_repeats=10, random_state=42, n_jobs=-1
)

# نحول النتائج لسيريز ونرتبها
importances = pd.Series(result.importances_mean, index=clean_names)
importances = importances.sort_values(ascending=False)

# عرض أهم 10 ميزات
top_features = importances.head(20)
print("\nTop 20 Features by Permutation Importance:")
print(top_features)

# ========= 5. Visualization ==========
plt.figure(figsize=(10, 6))
top_features.plot(kind='bar', color='green')
plt.title('Top 10 Most Important Features (Permutation Importance)')
plt.ylabel('Importance')
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
for col in top_features.index:
    if col in X.columns:
        if X[col].isna().sum() > 0:
            print(f"{col}: {X[col].isna().sum()}")

In [ ]:
plt.figure(figsize=(8,5))
X["GarageYrBlt"].hist(bins=30, color="green", edgecolor="black")
plt.title("توزيع GarageYrBlt قبل التعويض")
plt.xlabel("GarageYrBlt")
plt.ylabel("Count")
plt.show()


In [ ]:
X["GarageYrBlt"].fillna(X["GarageYrBlt"].mean(), inplace=True)

In [ ]:
plt.figure(figsize=(8,5))
X["GarageYrBlt"].hist(bins=30, color="green", edgecolor="black")
plt.title("توزيع GarageYrBlt بعد التعويض بالـ mean")
plt.xlabel("GarageYrBlt")
plt.ylabel("Count")
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression
#from sklearn.pipeline import Pipeline
#from sklearn.compose import ColumnTransformer
#from sklearn.preprocessing import OneHotEncoder, StandardScaler
#from sklearn.impute import SimpleImputer
#from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
#import matplotlib.pyplot as plt
#import numpy as np

# ===== 1. تصحيح أسماء الأعمدة =====
# الأعمدة العددية من top_features
num_features = [col for col in top_features.index if col in X.select_dtypes(include=np.number).columns]

# الأعمدة الفئوية الأصلية من top_features
cat_features = []
for col in top_features.index:
    for orig_col in cat_cols:  # الأعمدة الفئوية الأصلية
        if col.startswith(orig_col):
            cat_features.append(orig_col)

# دمجهم وإزالة التكرارات
top_features_original = list(set(num_features + cat_features))
X_top = X[top_features_original]

# ===== 2. تقسيم البيانات =====
X_train_top, X_test_top, y_train, y_test = train_test_split(
    X_top, y, test_size=0.2, random_state=42
)

# ===== 3. بايبلاين المعالجة =====
num_cols_top = [col for col in top_features_original if col in X.select_dtypes(include=np.number).columns]
cat_cols_top = [col for col in top_features_original if col in X.select_dtypes(include=['object', 'category']).columns]

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_top = ColumnTransformer([
    ('num', num_pipe, num_cols_top),
    ('cat', cat_pipe, cat_cols_top)
])

# ===== 4. بناء الموديل =====
lin_model = Pipeline([
    ('preprocessor', preprocessor_top),
    ('lr', LinearRegression())
])

# ===== 5. تدريب الموديل =====
lin_model.fit(X_train_top, y_train)

# ===== 6. التنبؤ =====
y_train_pred = lin_model.predict(X_train_top)
y_test_pred = lin_model.predict(X_test_top)

# ===== 7. حساب المقاييس =====
def print_metrics(y_true, y_pred, label=""):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"===== {label} Performance =====")
    print(f"MSE  : {mse:.2f}")
    print(f"RMSE : {rmse:.2f}")
    print(f"MAE  : {mae:.2f}")
    print(f"R²   : {r2:.4f}\n")

print_metrics(y_train, y_train_pred, "Training")
print_metrics(y_test, y_test_pred, "Testing")

# ===== 8. رسم scatter =====
plt.figure(figsize=(7,7))
plt.scatter(y_test, y_test_pred, color='blue', alpha=0.6, edgecolor='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel("القيم الحقيقية")
plt.ylabel("تنبؤات الموديل")
plt.title("LinearRegression: القيم الحقيقية vs التنبؤات")
plt.show()


In [ ]:
from sklearn.ensemble import RandomForestRegressor

# ===== 1. بايبلاين RandomForest =====
rf_model = Pipeline([
    ('preprocessor', preprocessor_top),  # نفس البايبلاين للمعالجة
    ('rf', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
])

# ===== 2. تدريب الموديل =====
rf_model.fit(X_train_top, y_train)

# ===== 3. التنبؤ =====
y_train_pred_rf = rf_model.predict(X_train_top)
y_test_pred_rf = rf_model.predict(X_test_top)

# ===== 4. حساب المقاييس =====
print_metrics(y_train, y_train_pred_rf, "Training (RandomForest)")
print_metrics(y_test, y_test_pred_rf, "Testing (RandomForest)")

# ===== 5. رسم scatter =====
plt.figure(figsize=(7,7))
plt.scatter(y_test, y_test_pred_rf, color='purple', alpha=0.6, edgecolor='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel("القيم الحقيقية")
plt.ylabel("تنبؤات RandomForest")
plt.title("RandomForest: القيم الحقيقية vs التنبؤات")
plt.show()
